# Level 2: performance tuning

Measures before and after for warehouse size, clustering, result cache, and a pre-aggregated gold table. Optional Preview: metric view materialization. Use warehouse `YOUR_PERF_WAREHOUSE_ID` only if your admin created a Medium warehouse for this lab.


## Bind config


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("telemetry_rows", "10000000", "Telemetry rows (10000000 or 50000000)")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)


## Baseline query times on fct_line_shift and telemetry sample

Compare these numbers after you change warehouse size in the SQL editor, not by rewriting ids into the notebook.


In [ ]:
import time

def timed(sql: str) -> float:
    t0 = time.time()
    spark.sql(sql).collect()
    return time.time() - t0

gold_s = timed(f"SELECT COUNT(*), SUM(good_units) FROM {cfg.table('fct_line_shift')}")
tel_s = timed(f"SELECT line_id, COUNT(*) FROM {cfg.table('sensor_telemetry')} GROUP BY line_id")
print("gold_seconds", gold_s)
print("telemetry_group_seconds", tel_s)
spark.sql(f"OPTIMIZE {cfg.table('sensor_telemetry')}")
tel_2 = timed(f"SELECT line_id, COUNT(*) FROM {cfg.table('sensor_telemetry')} GROUP BY line_id")
print("telemetry_after_optimize_seconds", tel_2)
print("PASS captured before/after optimize. Repeat in the UI on Small vs Medium warehouses.")
